# FahMai RAG V1 BM25 Baseline

First non-LLM baseline for the FahMai Level 1 challenge.

What this notebook does:

- loads Thai markdown knowledge-base files
- parses them with `markdown-it-py`
- converts them into retrieval chunks
- builds a BM25 index with Thai-safe mixed tokenization
- scores answer choices `1`-`8`
- applies explicit fallback logic for `9` and `10`
- writes a valid submission CSV


In [1]:
from __future__ import annotations

import math
import os
import re
import time
from collections import Counter
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display
from markdown_it import MarkdownIt
from rank_bm25 import BM25Okapi

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 220)


In [2]:
NOTEBOOK_SLUG = "rag-v1-bm25"
TOP_K = 8
MIN_CHUNK_LEN = 80
QUESTION_WEAK_SCORE = 1.25
CHOICE_WEAK_SCORE = 1.10
OUT_OF_DOMAIN_LENGTH = 120
OUTPUT_DIRNAME = "outputs"

KNOWN_STORE_TERMS = {
    "ฟ้าใหม่",
    "fahmai",
    "สายฟ้า",
    "saifah",
    "ดาวเหนือ",
    "daonuea",
    "คลื่นเสียง",
    "kluensiang",
    "ว่องโคจร",
    "wongkhojon",
    "จัดชุ่ม",
    "judchuam",
    "เซนไบต์",
    "zenbyte",
    "novatech",
    "arcwave",
    "pulsegear",
}

OUT_OF_DOMAIN_HINTS = {
    "hotel",
    "resort",
    "restaurant",
    "beach",
    "phuket",
    "flight",
    "island",
    "freedom",
    "travel",
    "โรงแรม",
    "ร้านอาหาร",
    "ภูเก็ต",
    "ป่าตอง",
    "ทะเล",
    "ดำน้ำ",
    "เที่ยว",
    "เรือ",
    "รีสอร์ท",
}


def resolve_project_dir() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "data" / "knowledge_base").exists() and (
            candidate / "pyproject.toml"
        ).exists():
            return candidate
    raise FileNotFoundError("Could not resolve project directory")


PROJECT_DIR = resolve_project_dir()
DATA_DIR = PROJECT_DIR / "data"
KB_DIR = DATA_DIR / "knowledge_base"
QUESTIONS_PATH = DATA_DIR / "questions.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"
OUTPUT_DIR = PROJECT_DIR / OUTPUT_DIRNAME
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / f"submission_{NOTEBOOK_SLUG}.csv"

print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"KB_DIR: {KB_DIR}")
print(f"QUESTIONS_PATH: {QUESTIONS_PATH}")
print(f"OUTPUT_PATH: {OUTPUT_PATH}")


PROJECT_DIR: /Users/beam/Workspace/Project/my-kaggle-notebooks/notebooks/2026-03-27-super-ai-engineer-s6-fahmai-rag-challenge-level-1
KB_DIR: /Users/beam/Workspace/Project/my-kaggle-notebooks/notebooks/2026-03-27-super-ai-engineer-s6-fahmai-rag-challenge-level-1/data/knowledge_base
QUESTIONS_PATH: /Users/beam/Workspace/Project/my-kaggle-notebooks/notebooks/2026-03-27-super-ai-engineer-s6-fahmai-rag-challenge-level-1/data/questions.csv
OUTPUT_PATH: /Users/beam/Workspace/Project/my-kaggle-notebooks/notebooks/2026-03-27-super-ai-engineer-s6-fahmai-rag-challenge-level-1/outputs/submission_rag-v1-bm25.csv


In [3]:
def normalize_whitespace(text: str) -> str:
    text = text.replace("\xa0", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


THAI_CHAR_RE = re.compile(r"[ก-๙]")
ALNUM_TOKEN_RE = re.compile(r"[a-z0-9][a-z0-9.+/-]*")
NUMBER_TOKEN_RE = re.compile(r"\d+(?:[.,]\d+)?")
THAI_WORDLIKE_RE = re.compile(r"[ก-๙]+")


def normalize_text(text: str) -> str:
    text = text.lower()
    text = text.replace("–", "-").replace("—", "-")
    text = text.replace("×", "x")
    text = re.sub(r"[^0-9a-zก-๙.%/+-]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def thai_char_ngrams(text: str, n: int = 3) -> list[str]:
    compact = re.sub(r"[^ก-๙]", "", text)
    if len(compact) < n:
        return [compact] if compact else []
    return [compact[i : i + n] for i in range(len(compact) - n + 1)]


def tokenize_for_bm25(text: str) -> list[str]:
    normalized = normalize_text(text)
    ascii_tokens = ALNUM_TOKEN_RE.findall(normalized)
    thai_wordlike = THAI_WORDLIKE_RE.findall(normalized)
    thai_ngrams = []
    for token in thai_wordlike:
        thai_ngrams.extend(thai_char_ngrams(token, n=3))
    tokens = ascii_tokens + thai_ngrams
    return tokens or [normalized] if normalized else []


def extract_support_tokens(text: str) -> set[str]:
    normalized = normalize_text(text)
    parts = set(ALNUM_TOKEN_RE.findall(normalized))
    parts.update(NUMBER_TOKEN_RE.findall(normalized))
    parts.update(
        token for token in THAI_WORDLIKE_RE.findall(normalized) if len(token) >= 2
    )
    return {token for token in parts if token}


def load_markdown_corpus(root: Path) -> list[dict[str, Any]]:
    records = []
    for path in sorted(root.rglob("*.md")):
        text = path.read_text(encoding="utf-8")
        records.append(
            {
                "source_path": str(path.relative_to(root.parent)),
                "doc_type": path.parent.name,
                "filename": path.name,
                "path": path,
                "text": text,
                "char_count": len(text),
            }
        )
    return records


corpus = load_markdown_corpus(KB_DIR)
questions_df = pd.read_csv(QUESTIONS_PATH)
sample_submission_df = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print(f"Loaded {len(corpus)} markdown files")
print(f"Loaded {len(questions_df)} questions")
print(f"Sample submission shape: {sample_submission_df.shape}")


Loaded 118 markdown files
Loaded 100 questions
Sample submission shape: (100, 2)


In [4]:
def parse_with_markdown_it(text: str, source_path: str) -> dict[str, Any]:
    start = time.perf_counter()
    md = MarkdownIt("commonmark").enable("table").enable("strikethrough")
    lines = text.splitlines()
    tokens = md.parse(text)
    stack: list[str | None] = []
    headings: list[str] = []
    blocks: list[dict[str, Any]] = []

    def current_path() -> list[str]:
        return [item for item in stack if item]

    i = 0
    while i < len(tokens):
        token = tokens[i]
        if token.type == "heading_open":
            level = int(token.tag[1])
            inline_token = tokens[i + 1] if i + 1 < len(tokens) else None
            heading_text = normalize_whitespace(getattr(inline_token, "content", ""))
            while len(stack) < level:
                stack.append(None)
            stack[level - 1] = heading_text
            stack[:] = stack[:level]
            headings.append(heading_text)
            i += 1
            i += 1
            continue
        if token.type in {
            "paragraph_open",
            "blockquote_open",
            "bullet_list_open",
            "ordered_list_open",
            "table_open",
        }:
            start_line, end_line = token.map or (None, None)
            segment = (
                "\n".join(lines[start_line:end_line])
                if start_line is not None and end_line is not None
                else ""
            )
            if segment:
                blocks.append(
                    {
                        "block_type": token.type.removesuffix("_open"),
                        "heading_path": current_path(),
                        "text": normalize_whitespace(segment),
                        "contains_table": token.type == "table_open",
                    }
                )
        i += 1

    return {
        "source_path": source_path,
        "doc_type": Path(source_path).parts[1],
        "headings": headings,
        "blocks": blocks,
        "parse_time_ms": round((time.perf_counter() - start) * 1000, 3),
    }


def merge_short_chunks(
    blocks: list[dict[str, Any]], min_len: int = MIN_CHUNK_LEN
) -> list[dict[str, Any]]:
    merged: list[dict[str, Any]] = []
    for block in blocks:
        if not merged:
            merged.append(block.copy())
            continue
        previous = merged[-1]
        same_heading = previous["heading_path"] == block["heading_path"]
        same_table_flag = previous["contains_table"] == block["contains_table"]
        if same_heading and same_table_flag and len(previous["text"]) < min_len:
            previous["text"] = normalize_whitespace(
                previous["text"] + "\n\n" + block["text"]
            )
        else:
            merged.append(block.copy())
    return merged


def filename_tokens(source_path: str) -> str:
    name = Path(source_path).stem.lower()
    return name.replace("_", " ").replace("-", " ")


def to_rag_chunks(parsed_doc: dict[str, Any]) -> list[dict[str, Any]]:
    merged_blocks = merge_short_chunks(parsed_doc["blocks"])
    chunks = []
    for idx, block in enumerate(merged_blocks):
        heading_path = " > ".join(block["heading_path"])
        retrieval_text = normalize_whitespace(
            "\n".join(
                part
                for part in [
                    heading_path,
                    block["text"],
                    filename_tokens(parsed_doc["source_path"]),
                ]
                if part
            )
        )
        chunks.append(
            {
                "source_path": parsed_doc["source_path"],
                "doc_type": parsed_doc["doc_type"],
                "heading_path": heading_path,
                "chunk_text": block["text"],
                "retrieval_text": retrieval_text,
                "chunk_order": idx,
                "contains_table": block["contains_table"],
            }
        )
    return chunks


In [5]:
parsed_docs = [
    parse_with_markdown_it(record["text"], record["source_path"]) for record in corpus
]
all_chunks = []
for parsed_doc in parsed_docs:
    all_chunks.extend(to_rag_chunks(parsed_doc))
chunks_df = pd.DataFrame(all_chunks)

print(f"Parsed docs: {len(parsed_docs)}")
print(f"Chunks: {len(chunks_df)}")
display(chunks_df.head())


Parsed docs: 118
Chunks: 2273


,source_path,doc_type,heading_path,chunk_text,retrieval_text,chunk_order,contains_table
0,knowledge_base/policies/cancellation_policy.md,policies,นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่,**วันที่อัปเดต:** 1 มีนาคม 2569,นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่\n**วันที่อัปเดต:** 1 มีนาคม 2569\ncancellation policy,0,False
1,knowledge_base/policies/cancellation_policy.md,policies,นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 1. ภาพรวมนโยบาย,ฟ้าใหม่เข้าใจว่าบางครั้งลูกค้าอาจต้องการยกเลิกคำสั่งซื้อด้วยเหตุผลต่างๆ นโยบายนี้อธิบายสิทธิ์และขั้นตอนการยกเลิกคำสั่งซื้อตามสถานะของคำสั่งซื้อในขณะนั้น ความสามารถในการยกเลิกขึ...,นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 1. ภาพรวมนโยบาย\nฟ้าใหม่เข้าใจว่าบางครั้งลูกค้าอาจต้องการยกเลิกคำสั่งซื้อด้วยเหตุผลต่างๆ นโยบายนี้อธิบายสิทธิ์และขั้นตอนการยกเลิกคำสั่...,1,False
2,knowledge_base/policies/cancellation_policy.md,policies,"นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 2. การยกเลิกตามสถานะคำสั่งซื้อ > 2.1 สถานะ ""รอชำระเงิน"" (Pending Payment)",**ยกเลิกได้ทันที**\n\nคำสั่งซื้อที่อยู่ในสถานะรอชำระเงินสามารถยกเลิกได้ทันทีโดยไม่มีค่าใช้จ่าย ผ่านแอปพลิเคชัน FahMai หรือเว็บไซต์ www.fahmai.th,"นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 2. การยกเลิกตามสถานะคำสั่งซื้อ > 2.1 สถานะ ""รอชำระเงิน"" (Pending Payment)\n**ยกเลิกได้ทันที**\n\nคำสั่งซื้อที่อยู่ในสถานะรอชำระเงินสาม...",2,False
3,knowledge_base/policies/cancellation_policy.md,policies,"นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 2. การยกเลิกตามสถานะคำสั่งซื้อ > 2.1 สถานะ ""รอชำระเงิน"" (Pending Payment)",**Auto-cancel:** หากไม่มีการชำระเงินภายใน **24 ชั่วโมง** นับจากเวลาที่สร้างคำสั่งซื้อ ระบบจะยกเลิกคำสั่งซื้อโดยอัตโนมัติ และสินค้าจะกลับไปยังคลังสินค้า,"นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 2. การยกเลิกตามสถานะคำสั่งซื้อ > 2.1 สถานะ ""รอชำระเงิน"" (Pending Payment)\n**Auto-cancel:** หากไม่มีการชำระเงินภายใน **24 ชั่วโมง** นั...",3,False
4,knowledge_base/policies/cancellation_policy.md,policies,"นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 2. การยกเลิกตามสถานะคำสั่งซื้อ > 2.1 สถานะ ""รอชำระเงิน"" (Pending Payment)",> ไม่มีการหักค่าธรรมเนียมใดๆ สำหรับการยกเลิกในสถานะนี้\n\n> ไม่มีการหักค่าธรรมเนียมใดๆ สำหรับการยกเลิกในสถานะนี้,"นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 2. การยกเลิกตามสถานะคำสั่งซื้อ > 2.1 สถานะ ""รอชำระเงิน"" (Pending Payment)\n> ไม่มีการหักค่าธรรมเนียมใดๆ สำหรับการยกเลิกในสถานะนี้\n\n>...",4,False


In [6]:
def build_bm25_index(chunks_df: pd.DataFrame) -> tuple[BM25Okapi, list[list[str]]]:
    tokenized_corpus = [
        tokenize_for_bm25(text) for text in chunks_df["retrieval_text"].tolist()
    ]
    index = BM25Okapi(tokenized_corpus)
    return index, tokenized_corpus


bm25_index, tokenized_corpus = build_bm25_index(chunks_df)

question_terms = set()
for question in questions_df["question"]:
    question_terms.update(extract_support_tokens(str(question)))

known_product_terms = set()
for source_path in chunks_df["source_path"].tolist():
    known_product_terms.update(
        token for token in filename_tokens(source_path).split() if len(token) >= 3
    )

DOMAIN_TERMS = KNOWN_STORE_TERMS | known_product_terms | question_terms


def retrieve_chunks(query: str, top_k: int = TOP_K) -> pd.DataFrame:
    query_tokens = tokenize_for_bm25(query)
    scores = bm25_index.get_scores(query_tokens)
    ranked_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[
        :top_k
    ]
    result = chunks_df.iloc[ranked_idx].copy()
    result["bm25_score"] = [float(scores[i]) for i in ranked_idx]
    return result.reset_index(drop=True)


def support_bonus(choice_text: str, chunk_text: str) -> float:
    choice_tokens = extract_support_tokens(choice_text)
    chunk_tokens = extract_support_tokens(chunk_text)
    if not choice_tokens or not chunk_tokens:
        return 0.0
    overlap = choice_tokens & chunk_tokens
    if not overlap:
        return 0.0
    overlap_ratio = len(overlap) / max(len(choice_tokens), 1)
    numeric_overlap = sum(1 for token in overlap if NUMBER_TOKEN_RE.fullmatch(token))
    table_bonus = 0.15 if "|" in chunk_text else 0.0
    return overlap_ratio * 0.8 + numeric_overlap * 0.2 + table_bonus


def looks_store_related(question_text: str, top_chunks: pd.DataFrame) -> bool:
    normalized = normalize_text(question_text)
    if any(term in normalized for term in DOMAIN_TERMS if len(term) >= 2):
        return True
    return bool((top_chunks["bm25_score"] > QUESTION_WEAK_SCORE).any())


def looks_out_of_domain(question_text: str, top_chunks: pd.DataFrame) -> bool:
    normalized = normalize_text(question_text)
    weak_retrieval = (
        top_chunks["bm25_score"].max() < QUESTION_WEAK_SCORE
        if not top_chunks.empty
        else True
    )
    has_hint = any(term in normalized for term in OUT_OF_DOMAIN_HINTS)
    long_story = len(question_text) >= OUT_OF_DOMAIN_LENGTH
    has_store_term = any(term in normalized for term in DOMAIN_TERMS if len(term) >= 2)
    return weak_retrieval and (has_hint or long_story) and not has_store_term


In [7]:
def score_choices(question_row: pd.Series, top_k: int = TOP_K) -> dict[str, Any]:
    question_text = str(question_row["question"])
    question_top_chunks = retrieve_chunks(question_text, top_k=top_k)

    choice_rows = []
    for answer_id in range(1, 9):
        choice_text = str(question_row[f"choice_{answer_id}"])
        choice_query = normalize_whitespace(question_text + "\n" + choice_text)
        ranked = retrieve_chunks(choice_query, top_k=top_k)
        if ranked.empty:
            choice_rows.append(
                {
                    "answer_id": answer_id,
                    "choice_text": choice_text,
                    "score": -1e9,
                    "best_chunk": "",
                    "best_source_path": "",
                }
            )
            continue
        ranked = ranked.copy()
        ranked["support_bonus"] = ranked["chunk_text"].map(
            lambda chunk: support_bonus(choice_text, chunk)
        )
        ranked["final_score"] = ranked["bm25_score"] + ranked["support_bonus"]
        best_row = ranked.sort_values(
            ["final_score", "contains_table"], ascending=[False, False]
        ).iloc[0]
        choice_rows.append(
            {
                "answer_id": answer_id,
                "choice_text": choice_text,
                "score": float(best_row["final_score"]),
                "bm25_score": float(best_row["bm25_score"]),
                "support_bonus": float(best_row["support_bonus"]),
                "best_chunk": best_row["chunk_text"],
                "best_source_path": best_row["source_path"],
                "best_heading_path": best_row["heading_path"],
            }
        )

    choice_scores_df = (
        pd.DataFrame(choice_rows)
        .sort_values(["score", "answer_id"], ascending=[False, True])
        .reset_index(drop=True)
    )
    best_choice = choice_scores_df.iloc[0]
    question_top_score = (
        float(question_top_chunks["bm25_score"].max())
        if not question_top_chunks.empty
        else 0.0
    )
    store_related = looks_store_related(question_text, question_top_chunks)
    out_of_domain = looks_out_of_domain(question_text, question_top_chunks)

    if best_choice["score"] < CHOICE_WEAK_SCORE:
        predicted_answer = 10 if out_of_domain or not store_related else 9
    else:
        predicted_answer = int(best_choice["answer_id"])

    return {
        "id": int(question_row["id"]),
        "question": question_text,
        "predicted_answer": predicted_answer,
        "best_choice_id": int(best_choice["answer_id"]),
        "best_choice_score": float(best_choice["score"]),
        "question_top_score": question_top_score,
        "store_related": bool(store_related),
        "out_of_domain": bool(out_of_domain),
        "choice_scores_df": choice_scores_df,
        "question_top_chunks": question_top_chunks,
    }


def predict_all_questions(
    questions_df: pd.DataFrame,
) -> tuple[pd.DataFrame, list[dict[str, Any]]]:
    prediction_rows = []
    debug_rows = []
    for _, row in questions_df.iterrows():
        result = score_choices(row)
        prediction_rows.append(
            {"id": result["id"], "answer": result["predicted_answer"]}
        )
        debug_rows.append(
            {
                "id": result["id"],
                "question": result["question"],
                "answer": result["predicted_answer"],
                "best_choice_id": result["best_choice_id"],
                "best_choice_score": result["best_choice_score"],
                "question_top_score": result["question_top_score"],
                "store_related": result["store_related"],
                "out_of_domain": result["out_of_domain"],
            }
        )
    return pd.DataFrame(prediction_rows), debug_rows


predictions_df, debug_rows = predict_all_questions(questions_df)
retrieval_debug_df = pd.DataFrame(debug_rows)
display(retrieval_debug_df.head(10))


,id,question,answer,best_choice_id,best_choice_score,question_top_score,store_related,out_of_domain
0,1,Watch S3 Ultra กันน้ำได้กี่ ATM ครับ,5,5,58.391620,48.284782,True,False
1,2,ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ,3,3,52.163081,45.333713,True,False
2,3,หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ,8,8,32.063330,23.294664,True,False
3,4,อยากเอามือถือเครื่องเก่ามาเทิร์น เปลี่ยนเป็นเครื่องใหม่ที่ฟ้าใหม่ได้ไหม?,6,6,126.276305,68.193199,True,False
4,5,สั่งของจากฟ้าใหม่ สั่งได้ครั้งละกี่ชิ้นครับ,6,6,98.528733,87.894933,True,False
5,6,ฟ้าใหม่จ่ายด้วย crypto ได้มั้ยคะ เช่น Bitcoin,8,8,137.041268,21.430142,True,False
6,7,ซื้อ AirBook 14 มาแล้ว อยากรู้ว่าในกล่องมีอะไรมาบ้าง,6,6,104.117784,37.015820,True,False
7,8,DuoPad สั่งซื้อได้เลยไหมครับ หรือต้องพรีออเดอร์,4,4,99.517552,45.397987,True,False
8,9,เฮ้ยย อ่านเยอะมากเลย ขอถามนิดนึงนะคะ คือพอดีว่าจะไปเที่ยวภูเก็ตกับเพื่อนสนิท 3 คนช่วงวันหยุดยาวเดือนหน้า จองโรงแรม Kata Beach Resort ไว้แล้ว 3 คืน เพื่อนบอกว่าตรงนั้นมีจุดดำน้ำ...,1,1,281.316216,203.526578,True,False
9,10,อยากใช้ซิม 2 ค่ายพร้อมกันครับ ใส่ซิมได้ 2 ใบมั้ย ไม่อยากพกมือถือ 2 เครื่อง มองอยู่รุ่น X9 Pro,3,3,174.349791,82.553813,True,False


In [8]:
interesting_ids = [1, 2, 3, 4, 6, 9, 38, 39]
for question_id in interesting_ids:
    result = score_choices(questions_df.loc[questions_df["id"] == question_id].iloc[0])
    print("=" * 120)
    print(f"ID {question_id}: {result['question']}")
    print(f"Predicted answer: {result['predicted_answer']}")
    print(f"Best choice id: {result['best_choice_id']}")
    print(f"Best choice score: {result['best_choice_score']:.4f}")
    print(f"Question top score: {result['question_top_score']:.4f}")
    print(
        f"Store related: {result['store_related']} | Out of domain: {result['out_of_domain']}"
    )
    print("Top choice candidates:")
    display(result["choice_scores_df"].head(3))
    print("Top retrieved chunks:")
    display(
        result["question_top_chunks"][
            ["source_path", "heading_path", "chunk_text", "bm25_score"]
        ].head(5)
    )


ID 1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
Predicted answer: 5
Best choice id: 5
Best choice score: 58.3916
Question top score: 48.2848
Store related: True | Out of domain: False
Top choice candidates:


,answer_id,choice_text,score,bm25_score,support_bonus,best_chunk,best_source_path,best_heading_path
0,5,10 ATM,58.391620,57.391620,1.0,**Q: ว่ายน้ำได้ไหมคะ? ดำน้ำได้ด้วยไหม?**\nA: ว่ายน้ำได้สบายเลยค่ะ กันน้ำ 5 ATM (50 เมตร) รองรับว่ายน้ำในสระและทะเล แต่ไม่มีโหมด Dive Mode สำหรับการดำน้ำลึก ถ้าต้องการ Dive Mode...,knowledge_base/products/WK-SW-002_wongkhojon_watch_s3_pro.md,วงโคจร Watch S3 Pro (WongKhoJon Watch S3 Pro) > คำถามที่พบบ่อยเกี่ยวกับสินค้านี้
1,3,5 ATM,57.891763,56.891763,1.0,**Q: ว่ายน้ำได้ไหมคะ? ดำน้ำได้ด้วยไหม?**\nA: ว่ายน้ำได้สบายเลยค่ะ กันน้ำ 5 ATM (50 เมตร) รองรับว่ายน้ำในสระและทะเล แต่ไม่มีโหมด Dive Mode สำหรับการดำน้ำลึก ถ้าต้องการ Dive Mode...,knowledge_base/products/WK-SW-002_wongkhojon_watch_s3_pro.md,วงโคจร Watch S3 Pro (WongKhoJon Watch S3 Pro) > คำถามที่พบบ่อยเกี่ยวกับสินค้านี้
2,1,3 ATM,55.481993,54.481993,1.0,**Q: ว่ายน้ำได้ไหมคะ? ดำน้ำได้ด้วยไหม?**\nA: ว่ายน้ำได้สบายเลยค่ะ กันน้ำ 5 ATM (50 เมตร) รองรับว่ายน้ำในสระและทะเล แต่ไม่มีโหมด Dive Mode สำหรับการดำน้ำลึก ถ้าต้องการ Dive Mode...,knowledge_base/products/WK-SW-002_wongkhojon_watch_s3_pro.md,วงโคจร Watch S3 Pro (WongKhoJon Watch S3 Pro) > คำถามที่พบบ่อยเกี่ยวกับสินค้านี้


Top retrieved chunks:


,source_path,heading_path,chunk_text,bm25_score
0,knowledge_base/products/WK-SW-002_wongkhojon_watch_s3_pro.md,วงโคจร Watch S3 Pro (WongKhoJon Watch S3 Pro) > คำถามที่พบบ่อยเกี่ยวกับสินค้านี้,**Q: ว่ายน้ำได้ไหมคะ? ดำน้ำได้ด้วยไหม?**\nA: ว่ายน้ำได้สบายเลยค่ะ กันน้ำ 5 ATM (50 เมตร) รองรับว่ายน้ำในสระและทะเล แต่ไม่มีโหมด Dive Mode สำหรับการดำน้ำลึก ถ้าต้องการ Dive Mode...,48.284782
1,knowledge_base/products/WK-SW-001_wongkhojon_watch_s3_ultra.md,วงโคจร Watch S3 Ultra (WongKhoJon Watch S3 Ultra) > คำถามที่พบบ่อยเกี่ยวกับสินค้านี้,**Q: Dive Mode ใช้ดำน้ำได้ลึกแค่ไหนครับ?**\nA: Dive Mode ของ S3 Ultra รองรับการดำน้ำสูงสุด 40 เมตร (ลึกกว่านั้นนาฬิกาจะไม่รับประกันความถูกต้องของข้อมูลและอาจเกิดความเสียหายได้)...,37.807190
2,knowledge_base/products/WK-SW-001_wongkhojon_watch_s3_ultra.md,วงโคจร Watch S3 Ultra (WongKhoJon Watch S3 Ultra) > รายละเอียดสินค้า,ความโดดเด่นที่สุดของ S3 Ultra คือมาตรฐานกันน้ำระดับ 100 เมตร (10 ATM) พร้อมโหมด Dive Mode ที่รองรับการดำน้ำได้ถึง 40 เมตร บันทึกความลึก ความดัน อุณหภูมิน้ำ และเวลาดำน้ำแบบ real...,37.424265
3,knowledge_base/products/WK-SW-001_wongkhojon_watch_s3_ultra.md,วงโคจร Watch S3 Ultra (WongKhoJon Watch S3 Ultra) > คำถามที่พบบ่อยเกี่ยวกับสินค้านี้,"**Q: Watch S3 Ultra ต่างจาก Watch S3 Pro ยังไงบ้างครับ?**\nA: S3 Ultra แตกต่างจาก S3 Pro ในหลายด้านครับ ได้แก่ (1) ตัวเรือนไทเทเนียม เบากว่าสแตนเลสของ S3 Pro, (2) กันน้ำ 100 เม...",35.102586
4,knowledge_base/products/WK-SW-004_wongkhojon_watch_s3_se.md,วงโคจร Watch S3 SE (WongKhoJon Watch S3 SE) > รายละเอียดสินค้า,**ข้อควรทราบสำคัญ:** Watch S3 SE ไม่มี GPS ในตัว (ใช้ GPS จากโทรศัพท์แทน — Connected GPS) ไม่มี ECG และไม่มี NFC Pay มาตรฐานกันน้ำ 3 ATM (30 เมตร) รองรับฝนและการล้างมือ แต่ไม่เ...,29.137918


ID 2: ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ
Predicted answer: 3
Best choice id: 3
Best choice score: 52.1631
Question top score: 45.3337
Store related: True | Out of domain: False
Top choice candidates:


...

,source_path,heading_path,chunk_text,bm25_score
0,knowledge_base/policies/shipping_policy.md,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่ > 3. ระยะเวลาจัดส่ง > 3.2 การจัดส่ง Express (กรุงเทพฯ + ปริมณฑลเท่านั้น),- สินค้ามีพร้อมในคลัง (ไม่ใช่ Pre-order)\n\nเพิ่มค่าบริการ +฿100 (ยกเว้นสมาชิก Platinum ซึ่ง Express ฟรี),72.415964
1,knowledge_base/policies/shipping_policy.md,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่ > 3. ระยะเวลาจัดส่ง > 3.2 การจัดส่ง Express (กรุงเทพฯ + ปริมณฑลเท่านั้น),- ที่อยู่จัดส่งอยู่ในเขตกรุงเทพมหานครหรือปริมณฑล\n\n- สั่งและชำระเงินก่อน 12:00 น. ในวันทำการ,72.314816
2,knowledge_base/policies/shipping_policy.md,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่ > 3. ระยะเวลาจัดส่ง > 3.2 การจัดส่ง Express (กรุงเทพฯ + ปริมณฑลเท่านั้น),บริการ Express จัดส่งภายใน **1 วันทำการ** สำหรับคำสั่งซื้อที่:\n\n- ที่อยู่จัดส่งอยู่ในเขตกรุงเทพมหานครหรือปริมณฑล\n- สั่งและชำระเงินก่อน 12:00 น. ในวันทำการ\n- สินค้ามีพร้อมใน...,68.596661
3,knowledge_base/policies/membership_points_policy.md,นโยบายสมาชิกและ FahMai Points — ร้านฟ้าใหม่ > 2. ระดับสมาชิก (Membership Tiers) > 2.3 สมาชิก Platinum,"| รายการ | รายละเอียด |\n|--------|-----------|\n| **เงื่อนไข** | ยอดสะสม ฿50,000 ขึ้นไป |\n| **อัตราสะสม Points** | **x2** (เก็บได้มากกว่า Silver 100%) |\n| **จัดส่งฟรี** | ทุ...",67.234135
4,knowledge_base/policies/shipping_policy.md,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่ > 2. ค่าจัดส่ง > 2.1 สมาชิก Gold และ Platinum,"- **Gold (ยอดสะสม ฿10,000–฿49,999):** จัดส่งฟรีทุกคำสั่งซื้อ ไม่มีขั้นต่ำ\n- **Platinum (ยอดสะสม ฿50,000 ขึ้นไป):** จัดส่งฟรี + Express ฟรี (ในเขตกรุงเทพฯ+ปริมณฑล)",60.481331


ID 39: สั่ง Pre-order สายฟ้า โฟน DuoPad ไว้ กำหนดส่งวันที่ 15 เมษายน ตอนนี้วันที่ 14 เมษายนแล้วครับ เปลี่ยนใจอยากยกเลิก ต้องเสียค่าอะไรมั้ย
Predicted answer: 1
Best choice id: 1
Best choice score: 311.2569
Question top score: 144.1356
Store related: True | Out of domain: False
Top choice candidates:


,answer_id,choice_text,score,bm25_score,support_bonus,best_chunk,best_source_path,best_heading_path
0,1,ยกเลิกได้ฟรีไม่เสียค่าใช้จ่าย เพราะยังไม่ถึงวันจัดส่ง 15 เมษายน สินค้า Pre-order ยกเลิกฟรีได้ทุกกรณี,311.256871,310.714014,0.542857,**ตัวอย่าง:** Pre-order กำหนดจัดส่งวันที่ 15 เมษายน ลูกค้ายกเลิกในวันที่ 11 เมษายน (เหลือ 4 วัน) → ยกเลิกฟรี ไม่มีค่าใช้จ่าย,knowledge_base/policies/cancellation_policy.md,นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 3. นโยบายการยกเลิกสินค้า Pre-order > 3.1 ยกเลิกก่อน 3 วันหรือมากกว่า ก่อนวันจัดส่ง — **ฟรี ไม่มีค่าใช้จ่าย**
1,4,ยกเลิกได้แต่จะถูกหักค่าดำเนินการ 5% ของราคาสินค้า เนื่องจากเหลือเวลาน้อยกว่า 3 วันก่อนวันจัดส่ง,302.998277,302.998277,0.000000,"**ตัวอย่าง:** Pre-order มูลค่า ฿10,000 ยกเลิกในวันที่ 13 เมษายน (กำหนดส่ง 15 เมษายน เหลือ 2 วัน)",knowledge_base/policies/cancellation_policy.md,นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 3. นโยบายการยกเลิกสินค้า Pre-order > 3.2 ยกเลิกน้อยกว่า 3 วัน ก่อนวันจัดส่ง — **หักค่าดำเนินการ 5%**
2,2,"ไม่สามารถยกเลิกได้เพราะเป็นสินค้า Pre-order ราคา 54,990 บาท ทุกคำสั่ง Pre-order ห้ามยกเลิกหลังชำระเงินแล้ว",232.788639,232.688639,0.100000,> **สินค้าสั่งจองล่วงหน้า** — กำหนดจัดส่งวันที่ 15 เมษายน 2569 ยกเลิกได้ตามนโยบายยกเลิกคำสั่งซื้อ Pre-order (ยกเลิกฟรีหากยกเลิกก่อน 3 วันก่อนจัดส่ง หักค่าดำเนินการ 5% หากยกเลิก...,knowledge_base/products/SF-SP-011_saifah_phone_duopad.md,สายฟ้า โฟน DuoPad (SaiFah Phone DuoPad) > หมายเหตุพิเศษ


Top retrieved chunks:


,source_path,heading_path,chunk_text,bm25_score
0,knowledge_base/products/SF-SP-011_saifah_phone_duopad.md,สายฟ้า โฟน DuoPad (SaiFah Phone DuoPad) > หมายเหตุพิเศษ,> **สินค้าสั่งจองล่วงหน้า** — กำหนดจัดส่งวันที่ 15 เมษายน 2569 ยกเลิกได้ตามนโยบายยกเลิกคำสั่งซื้อ Pre-order (ยกเลิกฟรีหากยกเลิกก่อน 3 วันก่อนจัดส่ง หักค่าดำเนินการ 5% หากยกเลิก...,144.135596
1,knowledge_base/products/SF-SP-011_saifah_phone_duopad.md,สายฟ้า โฟน DuoPad (SaiFah Phone DuoPad) > หมายเหตุพิเศษ,> **สินค้าสั่งจองล่วงหน้า** — กำหนดจัดส่งวันที่ 15 เมษายน 2569 ยกเลิกได้ตามนโยบายยกเลิกคำสั่งซื้อ Pre-order (ยกเลิกฟรีหากยกเลิกก่อน 3 วันก่อนจัดส่ง หักค่าดำเนินการ 5% หากยกเลิก...,144.135596
2,knowledge_base/policies/cancellation_policy.md,นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 3. นโยบายการยกเลิกสินค้า Pre-order > 3.2 ยกเลิกน้อยกว่า 3 วัน ก่อนวันจัดส่ง — **หักค่าดำเนินการ 5%**,"**ตัวอย่าง:** Pre-order มูลค่า ฿10,000 ยกเลิกในวันที่ 13 เมษายน (กำหนดส่ง 15 เมษายน เหลือ 2 วัน)",136.679022
3,knowledge_base/policies/cancellation_policy.md,นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 3. นโยบายการยกเลิกสินค้า Pre-order > 3.1 ยกเลิกก่อน 3 วันหรือมากกว่า ก่อนวันจัดส่ง — **ฟรี ไม่มีค่าใช้จ่าย**,**ตัวอย่าง:** Pre-order กำหนดจัดส่งวันที่ 15 เมษายน ลูกค้ายกเลิกในวันที่ 11 เมษายน (เหลือ 4 วัน) → ยกเลิกฟรี ไม่มีค่าใช้จ่าย,134.517244
4,knowledge_base/products/DN-LT-010_daonuea_stormbook_g9_titan.md,ดาวเหนือ สตอร์มบุ๊ก G9 Titan (DaoNuea StormBook G9 Titan) > หมายเหตุพิเศษ,> **สินค้าสั่งจองล่วงหน้า — กำหนดจัดส่งวันที่ 30 เมษายน 2569**\n>\n> StormBook G9 Titan อยู่ในช่วงสั่งจองล่วงหน้า สินค้าจะจัดส่งตามลำดับการสั่งจอง โดยมีกำหนดจัดส่งภายในวันที่ 3...,118.600890


In [9]:
submission_df = sample_submission_df.copy()
submission_df = submission_df[["id"]].merge(predictions_df, on="id", how="left")
assert len(submission_df) == len(sample_submission_df), "Submission row count mismatch"
assert submission_df["id"].tolist() == sample_submission_df["id"].tolist(), (
    "Submission ids changed order"
)
assert submission_df["answer"].between(1, 10).all(), (
    "Predicted answers must be in [1, 10]"
)
submission_df["answer"] = submission_df["answer"].astype(int)
submission_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved submission to {OUTPUT_PATH}")
print("Prediction distribution:")
display(submission_df["answer"].value_counts().sort_index().rename("rows").to_frame())
print("Weak-score rows:")
display(
    retrieval_debug_df.sort_values(["best_choice_score", "question_top_score"]).head(15)
)
display(submission_df.head(10))


Saved submission to /Users/beam/Workspace/Project/my-kaggle-notebooks/notebooks/2026-03-27-super-ai-engineer-s6-fahmai-rag-challenge-level-1/outputs/submission_rag-v1-bm25.csv
Prediction distribution:


,rows
answer,
1,18
2,13
3,10
4,12
5,11
6,16
7,10
8,10


Weak-score rows:


,id,question,answer,best_choice_id,best_choice_score,question_top_score,store_related,out_of_domain
2,3,หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ,8,8,32.063330,23.294664,True,False
53,54,สายฟ้า โฟน X9 Pro Max ผลิตที่ประเทศอะไรครับ?,5,5,34.340683,32.760424,True,False
61,62,ตอนนี้ดอกเบี้ยเงินฝากออมทรัพย์กี่เปอร์เซ็นต์คะ?,1,1,36.951232,36.951232,True,False
62,63,สูตรผัดกระเพราหมูสับทำยังไงคะ?,6,6,37.785842,27.209712,True,False
58,59,สายฟ้า โฟน X9 Pro Max ค่า SAR เท่าไหร่ครับ?,1,1,40.526950,40.526950,True,False
54,55,ฟ้าใหม่มีรายได้ต่อปีเท่าไหร่ครับ?,1,1,41.652834,40.526950,True,False
59,60,วันหยุดราชการปี 2569 มีกี่วันครับ?,1,1,42.364280,39.321356,True,False
56,57,สายฟ้า โฟน X9 Pro สัดส่วนหน้าจอต่อตัวเครื่อง (screen-to-body ratio) เท่าไหร่คะ?,1,1,43.985488,43.985488,True,False
1,2,ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ,3,3,52.163081,45.333713,True,False
52,53,เปลี่ยนจอ สายฟ้า โฟน X9 Pro ค่าซ่อมเท่าไหร่ครับ?,2,2,53.235581,47.954850,True,False


,id,answer
0,1,5
1,2,3
2,3,8
3,4,6
4,5,6
5,6,8
6,7,6
7,8,4
8,9,1
9,10,3
